# Governance-only generation notebook

This notebook contains only the Governance pipeline, split cell by cell for debugging.

It includes the Emirates-style Governance structure and the latest fixes:
- semi-annual Board reporting is separated from specific Board/committee decisions;
- climate risk register wording is evidence-anchored;
- deterministic risk-register checks scan the new Management responsibility subsection.


## 1. Setup, imports, environment variables and Azure OpenAI helpers


In [ ]:
# ============================================================
# GOVERNANCE SECTION GENERATOR
# Role-based Azure OpenAI REST endpoints
# Writer: GPT-5.1 | Judge: GPT-5.2 (LLM-only evaluation) | Reviser: GPT-4.1
# ============================================================

import os
import json
import re
import urllib.request
import urllib.error
import time
import random
from typing import TypedDict, Literal
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from langgraph.graph import StateGraph, START, END

# ── ENV LOADING ──────────────────────────────────────────────
env_path = find_dotenv()
if env_path:
    load_dotenv(env_path, override=True)
    print(f"Loaded .env from: {env_path}")
else:
    load_dotenv(override=True)
    print("No .env found by find_dotenv(); using existing environment variables.")

# Shared-key fallback.
# When all three deployments belong to the same Azure resource, one Azure key
# can authenticate all roles. The code also accepts any existing role-specific
# key as the shared fallback, which prevents unnecessary configuration failures.
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")

_shared_key_fallback = (
    AZURE_OPENAI_API_KEY
    or os.getenv("AZURE_OPENAI_WRITER_API_KEY")
    or os.getenv("AZURE_OPENAI_JUDGE_API_KEY")
    or os.getenv("AZURE_OPENAI_REVISER_API_KEY")
)

AZURE_OPENAI_WRITER_API_KEY = (
    os.getenv("AZURE_OPENAI_WRITER_API_KEY")
    or _shared_key_fallback
)
AZURE_OPENAI_JUDGE_API_KEY = (
    os.getenv("AZURE_OPENAI_JUDGE_API_KEY")
    or _shared_key_fallback
)
AZURE_OPENAI_REVISER_API_KEY = (
    os.getenv("AZURE_OPENAI_REVISER_API_KEY")
    or _shared_key_fallback
)

# Full Azure chat-completions deployment URLs.
# AZURE_OPENAI_CHAT_URL is retained as a backward-compatible writer fallback.
AZURE_OPENAI_WRITER_URL = (
    os.getenv("AZURE_OPENAI_WRITER_URL")
    or os.getenv("AZURE_OPENAI_CHAT_URL")
)
AZURE_OPENAI_JUDGE_URL = os.getenv("AZURE_OPENAI_JUDGE_URL")
AZURE_OPENAI_REVISER_URL = os.getenv("AZURE_OPENAI_REVISER_URL")


def _clean_url(value: str | None) -> str | None:
    if not value:
        return None
    return value.strip().strip('"').strip("'")


AZURE_OPENAI_WRITER_URL = _clean_url(AZURE_OPENAI_WRITER_URL)
AZURE_OPENAI_JUDGE_URL = _clean_url(AZURE_OPENAI_JUDGE_URL)
AZURE_OPENAI_REVISER_URL = _clean_url(AZURE_OPENAI_REVISER_URL)


def validate_role_config() -> None:
    required = {
        "AZURE_OPENAI_WRITER_API_KEY": AZURE_OPENAI_WRITER_API_KEY,
        "AZURE_OPENAI_JUDGE_API_KEY": AZURE_OPENAI_JUDGE_API_KEY,
        "AZURE_OPENAI_REVISER_API_KEY": AZURE_OPENAI_REVISER_API_KEY,
        "AZURE_OPENAI_WRITER_URL": AZURE_OPENAI_WRITER_URL,
        "AZURE_OPENAI_JUDGE_URL": AZURE_OPENAI_JUDGE_URL,
        "AZURE_OPENAI_REVISER_URL": AZURE_OPENAI_REVISER_URL,
    }

    missing = [name for name, value in required.items() if not value]
    if missing:
        loaded_flags = {
            "shared_key_loaded": bool(AZURE_OPENAI_API_KEY),
            "writer_key_loaded": bool(AZURE_OPENAI_WRITER_API_KEY),
            "judge_key_loaded": bool(AZURE_OPENAI_JUDGE_API_KEY),
            "reviser_key_loaded": bool(AZURE_OPENAI_REVISER_API_KEY),
            "writer_url_loaded": bool(AZURE_OPENAI_WRITER_URL),
            "judge_url_loaded": bool(AZURE_OPENAI_JUDGE_URL),
            "reviser_url_loaded": bool(AZURE_OPENAI_REVISER_URL),
        }
        raise ValueError(
            "Missing role-based Azure configuration values: "
            + ", ".join(missing)
            + "\n\nLoaded configuration flags (keys are never printed):\n"
            + json.dumps(loaded_flags, indent=2)
            + "\n\nRequired .env configuration when all deployments use the same Azure resource:\n"
              "AZURE_OPENAI_API_KEY=<shared Azure resource key>\n"
              "AZURE_OPENAI_WRITER_URL=<full GPT-5.1 deployment URL>\n"
              "AZURE_OPENAI_JUDGE_URL=<full GPT-5.2 deployment URL>\n"
              "AZURE_OPENAI_REVISER_URL=<full GPT-4.1 deployment URL>\n\n"
              "Use role-specific API keys only when a deployment belongs to a different Azure resource."
        )

    for name, url in {
        "AZURE_OPENAI_WRITER_URL": AZURE_OPENAI_WRITER_URL,
        "AZURE_OPENAI_JUDGE_URL": AZURE_OPENAI_JUDGE_URL,
        "AZURE_OPENAI_REVISER_URL": AZURE_OPENAI_REVISER_URL,
    }.items():
        if not url.startswith("https://"):
            raise ValueError(f"{name} must be a full HTTPS Azure deployment URL: {url!r}")


validate_role_config()

print("Role-based Azure OpenAI configuration loaded")
print("Writer endpoint (GPT-5.1):", AZURE_OPENAI_WRITER_URL[:90] + "...")
print("Judge endpoint  (GPT-5.2):", AZURE_OPENAI_JUDGE_URL[:90] + "...")
print("Reviser endpoint (GPT-4.1):", AZURE_OPENAI_REVISER_URL[:90] + "...")


def _azure_chat_completion(
    *,
    url: str,
    api_key: str,
    messages: list[dict],
    max_output_tokens: int,
    json_mode: bool = False,
    temperature: float | None = None,
    use_max_completion_tokens: bool = False,
    timeout: int = 240,
    request_label: str = "LLM",
    max_attempts: int = 4,
) -> dict:
    """
    Robust REST call for Azure/OpenAI-compatible enterprise gateways.

    Behaviour:
    - Retries transient 500/502/503/504 and connection errors.
    - For GPT-5.x gateways, first tries `max_completion_tokens`.
    - If the gateway returns 400/500, retries using `max_tokens`, because some
      enterprise proxies do not yet forward `max_completion_tokens` correctly.
    - Does not expose API keys in errors.
    """

    preferred_field = (
        "max_completion_tokens" if use_max_completion_tokens else "max_tokens"
    )
    token_fields = [preferred_field]
    if preferred_field == "max_completion_tokens":
        token_fields.append("max_tokens")

    last_error = None

    for token_field in token_fields:
        payload = {
            "messages": messages,
            token_field: max_output_tokens,
        }

        if temperature is not None:
            payload["temperature"] = temperature

        if json_mode:
            payload["response_format"] = {"type": "json_object"}

        for attempt in range(1, max_attempts + 1):
            req = urllib.request.Request(
                url,
                data=json.dumps(payload).encode("utf-8"),
                headers={
                    "Content-Type": "application/json",
                    "api-key": api_key,
                },
                method="POST",
            )

            try:
                with urllib.request.urlopen(req, timeout=timeout) as resp:
                    return json.loads(resp.read().decode("utf-8"))

            except urllib.error.HTTPError as exc:
                body = exc.read().decode(errors="replace")
                last_error = RuntimeError(
                    f"{request_label} HTTP error {exc.code}.\n"
                    f"Endpoint: {url}\n"
                    f"Token field used: {token_field}\n"
                    f"Response: {body[:3000]}"
                )

                transient = exc.code in {500, 502, 503, 504}
                compatibility_candidate = (
                    token_field == "max_completion_tokens"
                    and exc.code in {400, 422, 500}
                )

                if transient and attempt < max_attempts:
                    wait = min(2 ** (attempt - 1) + random.random(), 12)
                    print(
                        f"{request_label}: server error {exc.code}; "
                        f"retrying attempt {attempt + 1}/{max_attempts} "
                        f"in {wait:.1f}s..."
                    )
                    time.sleep(wait)
                    continue

                if compatibility_candidate:
                    print(
                        f"{request_label}: gateway may not support "
                        "`max_completion_tokens`; retrying with `max_tokens`."
                    )
                    break

                raise last_error from exc

            except urllib.error.URLError as exc:
                last_error = RuntimeError(
                    f"{request_label} connection error.\n"
                    f"Endpoint: {url!r}\n"
                    f"Error: {exc}"
                )

                if attempt < max_attempts:
                    wait = min(2 ** (attempt - 1) + random.random(), 12)
                    print(
                        f"{request_label}: connection issue; retrying "
                        f"attempt {attempt + 1}/{max_attempts} in {wait:.1f}s..."
                    )
                    time.sleep(wait)
                    continue

                raise last_error from exc

    raise last_error or RuntimeError(
        f"{request_label} request failed for an unknown reason."
    )

def _extract_message_content(data: dict) -> str:
    try:
        content = data["choices"][0]["message"]["content"]
    except (KeyError, IndexError, TypeError) as exc:
        raise ValueError(
            "Unexpected Azure response structure:\n"
            + json.dumps(data, indent=2, ensure_ascii=False)[:3000]
        ) from exc

    if not content:
        raise ValueError(
            "Azure returned an empty message content. Response:\n"
            + json.dumps(data, indent=2, ensure_ascii=False)[:3000]
        )
    return content


def call_writer_llm(system_prompt: str, user_prompt: str) -> str:
    """GPT-5.1 writer."""
    data = _azure_chat_completion(
        url=AZURE_OPENAI_WRITER_URL,
        api_key=AZURE_OPENAI_WRITER_API_KEY,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_output_tokens=2400,
        use_max_completion_tokens=True,
        temperature=None,
        request_label="GPT-5.1 writer",
    )
    return _extract_message_content(data)


def _extract_json_object(text: str) -> str:
    """
    Extract the outermost JSON object from model output.
    Handles accidental markdown fences or leading/trailing commentary.
    """
    text = text.strip()

    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
        text = re.sub(r"\s*```$", "", text)

    first = text.find("{")
    last = text.rfind("}")
    if first >= 0 and last > first:
        return text[first:last + 1]

    return text


def call_judge_llm_json(system_prompt: str, user_prompt: str) -> dict:
    """
    GPT-5.2 judge with JSON-safe retry logic.

    First call:
    - Requests valid JSON mode.
    - Uses a larger output allowance to avoid truncation.

    On invalid/truncated JSON:
    - Sends the returned content back to GPT-5.2 for JSON repair.
    - Requests a concise, complete JSON object only.
    """
    data = _azure_chat_completion(
        url=AZURE_OPENAI_JUDGE_URL,
        api_key=AZURE_OPENAI_JUDGE_API_KEY,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_output_tokens=2800,
        use_max_completion_tokens=True,
        temperature=None,
        json_mode=True,
        request_label="GPT-5.2 judge",
    )

    content = _extract_message_content(data)
    candidate = _extract_json_object(content)

    try:
        return json.loads(candidate)

    except json.JSONDecodeError:
        print("GPT-5.2 judge returned incomplete/invalid JSON. Attempting JSON repair...")

        repair_system = (
            "You repair malformed or truncated JSON. "
            "Return one complete valid JSON object only. "
            "Preserve the original meaning, scores, checklist values, issues, and fixes. "
            "Keep strings concise. Do not add markdown fences or commentary."
        )

        repair_user = f"""
Repair the following malformed or truncated judge output into one complete valid JSON object.

Requirements:
- Keep the same top-level fields when present.
- Finish incomplete strings and arrays conservatively.
- Limit each issue/fix string to at most 35 words.
- Limit arrays to the 6 most important items.
- Return JSON only.

MALFORMED OUTPUT:
{content}
""".strip()

        repaired_data = _azure_chat_completion(
            url=AZURE_OPENAI_JUDGE_URL,
            api_key=AZURE_OPENAI_JUDGE_API_KEY,
            messages=[
                {"role": "system", "content": repair_system},
                {"role": "user", "content": repair_user},
            ],
            max_output_tokens=2400,
            use_max_completion_tokens=True,
            temperature=None,
            json_mode=True,
            request_label="GPT-5.2 judge JSON repair",
        )

        repaired_content = _extract_message_content(repaired_data)
        repaired_candidate = _extract_json_object(repaired_content)

        try:
            return json.loads(repaired_candidate)
        except json.JSONDecodeError as exc:
            raise ValueError(
                "GPT-5.2 judge failed to return valid JSON even after repair.\n"
                f"Original output preview:\n{content[:4000]}\n\n"
                f"Repair output preview:\n{repaired_content[:4000]}"
            ) from exc


def call_reviser_llm(system_prompt: str, user_prompt: str) -> str:
    """GPT-4.1 reviser."""
    data = _azure_chat_completion(
        url=AZURE_OPENAI_REVISER_URL,
        api_key=AZURE_OPENAI_REVISER_API_KEY,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_output_tokens=2400,
        use_max_completion_tokens=False,
        temperature=0.1,
        request_label="GPT-4.1 reviser",
    )
    return _extract_message_content(data)




print("Governance LLM helper functions ready")


## 2. Load Governance payload and optional risk-register patch


In [ ]:
# ── LOAD GOVERNANCE PAYLOAD ─────────────────────────────────
# This standalone Governance notebook consumes:
#   payload_BANK01_governance.json
#
# Optional fallback patch:
# If the Governance payload does not include climate_risk_register, the notebook
# will copy it from payload_BANK01_risk_management.json or payload_BANK01_strategy.json
# when those files are available. This keeps the Governance management-responsibility
# disclosure evidence-based while avoiding risk-register invention.

def find_payload_file(filename: str) -> Path | None:
    search_dirs = [
        Path.cwd(),
        Path.cwd() / "Data",
        Path.cwd() / "payloads",
        Path.cwd().parent / "payloads",
        Path.cwd().parent / "Data",
        Path("/mnt/data"),
    ]
    for base in search_dirs:
        candidate = base / filename
        if candidate.exists():
            return candidate
    return None


def load_json_file(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


GOVERNANCE_PAYLOAD_PATH = (
    find_payload_file("payload_BANK01_governance.json")
    or find_payload_file("payload_BANK01.json")
)

OPTIONAL_RISK_MANAGEMENT_PAYLOAD_PATH = find_payload_file("payload_BANK01_risk_management.json")
OPTIONAL_STRATEGY_PAYLOAD_PATH = find_payload_file("payload_BANK01_strategy.json")

if GOVERNANCE_PAYLOAD_PATH is None:
    raise FileNotFoundError(
        "Could not find payload_BANK01_governance.json or payload_BANK01.json."
    )

governance_payload = load_json_file(GOVERNANCE_PAYLOAD_PATH)

risk_management_payload = {}
strategy_payload = {}

if OPTIONAL_RISK_MANAGEMENT_PAYLOAD_PATH is not None:
    risk_management_payload = load_json_file(OPTIONAL_RISK_MANAGEMENT_PAYLOAD_PATH)

if OPTIONAL_STRATEGY_PAYLOAD_PATH is not None:
    strategy_payload = load_json_file(OPTIONAL_STRATEGY_PAYLOAD_PATH)

# Patch climate_risk_register only if it is genuinely available in another section payload.
if "climate_risk_register" not in governance_payload:
    if "climate_risk_register" in risk_management_payload:
        governance_payload["climate_risk_register"] = risk_management_payload["climate_risk_register"]
        print("Patched Governance payload: added climate_risk_register from Risk Management payload.")
    elif "climate_risk_register" in strategy_payload:
        governance_payload["climate_risk_register"] = strategy_payload["climate_risk_register"]
        print("Patched Governance payload: added climate_risk_register from Strategy payload.")

# Backward-compatible alias used by the evidence extractor.
payload = governance_payload
PAYLOAD_PATH = GOVERNANCE_PAYLOAD_PATH

bank_name = governance_payload["bank"]["bank_name"]

print(f"Loaded Governance payload for: {bank_name}")
print(f"Governance payload path: {GOVERNANCE_PAYLOAD_PATH}")
print(f"Governance top-level keys: {list(governance_payload.keys())}")
print("Governance has climate_risk_register:", "climate_risk_register" in governance_payload)

## 3. Governance evidence extraction helpers


In [ ]:
# ── GOVERNANCE EVIDENCE EXTRACTOR ────────────────────────────
# Data-aware behaviour:
# - The Governance payload is the source of Governance facts.
# - This uploaded Governance payload contains governance, board_minutes and reporting_kpis.
# - It does not always contain climate_risk_register. If risk-register rows are absent,
#   the Management Responsibility subsection must be limited to the management committee,
#   board reporting frequency, ERM integration flag and major-transaction climate check.
# - The writer must not invent formal committee charters, trade-offs, escalation thresholds,
#   skills adequacy assessments, or assurance over financed emissions.

def _is_present(value) -> bool:
    return value is not None and str(value).strip().lower() not in {"", "nan", "none", "null"}


def _safe_int(value, default=0):
    try:
        return int(value)
    except Exception:
        return default


def _normalise_text(value) -> str:
    return re.sub(r"\s+", " ", str(value or "")).strip()


def _format_reporting_frequency(value: str | None) -> str | None:
    if not _is_present(value):
        return None
    return str(value).replace("_", "-").lower()


def extract_management_process_evidence(payload: dict, year: int = 2024) -> dict:
    """
    Summarise Management Responsibility evidence.

    Preferred source: climate_risk_register, when present.
    Fallback source: governance table fields in the Governance payload.
    """
    risks = [
        r for r in payload.get("climate_risk_register", [])
        if isinstance(r, dict) and _safe_int(r.get("reporting_year")) == year
    ]

    gov_records = payload.get("governance", [])
    gov_2024 = next(
        (
            r for r in gov_records
            if isinstance(r, dict) and _safe_int(r.get("reporting_year")) == year
        ),
        {},
    )

    governance_controls_available = any(
        _is_present(gov_2024.get(field))
        for field in [
            "management_committee_name",
            "climate_risk_reporting_to_board",
            "erm_integration_flag",
            "major_transactions_climate_check",
        ]
    )

    if not risks:
        return {
            "risk_register_available": False,
            "governance_controls_available": governance_controls_available,
            "reporting_year": year,
            "management_committee_name": gov_2024.get("management_committee_name"),
            "climate_risk_reporting_to_board": gov_2024.get("climate_risk_reporting_to_board"),
            "erm_integration_flag": gov_2024.get("erm_integration_flag"),
            "major_transactions_climate_check": gov_2024.get("major_transactions_climate_check"),
            "formal_escalation_thresholds_available": False,
            "message": (
                "The available Governance evidence does not contain climate_risk_register records. "
                "Management Responsibility can be described only using governance-level evidence."
            ),
            "process_flow_instruction": (
                "Do not claim a risk-register workflow, risk counts, risk categories, scenario links, "
                "monitoring frequencies by risk, or mitigation actions unless risk-register evidence is present. "
                "Use the available governance controls only: management committee name, board reporting frequency, "
                "ERM integration flag and major-transaction climate check."
            ),
        }

    frequencies = sorted({
        str(r.get("monitoring_frequency"))
        for r in risks
        if _is_present(r.get("monitoring_frequency"))
    })
    risk_categories = sorted({
        str(r.get("risk_category"))
        for r in risks
        if _is_present(r.get("risk_category"))
    })
    risk_ratings = sorted({
        str(r.get("risk_rating"))
        for r in risks
        if _is_present(r.get("risk_rating"))
    })
    scenario_links = sorted({
        str(r.get("scenario_analysis_link"))
        for r in risks
        if _is_present(r.get("scenario_analysis_link"))
    })
    mitigation_actions = sorted({
        str(r.get("mitigation_actions"))
        for r in risks
        if _is_present(r.get("mitigation_actions"))
    })

    integrated_count = sum(1 for r in risks if r.get("erm_integrated_flag") is True)
    changed_count = sum(1 for r in risks if r.get("changed_since_prior_period") is True)

    rating_priority = {"critical": 4, "high": 3, "medium": 2, "low": 1}
    sorted_risks = sorted(
        risks,
        key=lambda r: (
            rating_priority.get(str(r.get("risk_rating", "")).lower(), 0),
            float(r.get("financial_impact_meur") or 0),
        ),
        reverse=True,
    )

    material_risk_examples = []
    for r in sorted_risks[:5]:
        material_risk_examples.append({
            "risk_id": r.get("risk_id"),
            "risk_name": r.get("risk_name"),
            "risk_category": r.get("risk_category"),
            "risk_rating": r.get("risk_rating"),
            "time_horizon": r.get("time_horizon"),
            "financial_impact_meur": r.get("financial_impact_meur"),
            "monitoring_frequency": r.get("monitoring_frequency"),
            "erm_integrated_flag": r.get("erm_integrated_flag"),
            "scenario_analysis_link": r.get("scenario_analysis_link"),
            "mitigation_actions": r.get("mitigation_actions"),
        })

    return {
        "risk_register_available": True,
        "governance_controls_available": governance_controls_available,
        "reporting_year": year,
        "management_committee_name": gov_2024.get("management_committee_name"),
        "climate_risk_reporting_to_board": gov_2024.get("climate_risk_reporting_to_board"),
        "erm_integration_flag": gov_2024.get("erm_integration_flag"),
        "major_transactions_climate_check": gov_2024.get("major_transactions_climate_check"),
        "risk_count": len(risks),
        "erm_integrated_count": integrated_count,
        "changed_since_prior_period_count": changed_count,
        "monitoring_frequencies": frequencies,
        "risk_categories": risk_categories,
        "risk_ratings": risk_ratings,
        "scenario_analysis_links": scenario_links,
        "mitigation_actions": mitigation_actions[:8],
        "material_risk_examples": material_risk_examples,
        "formal_escalation_thresholds_available": False,
        "process_flow_instruction": (
            "Write management responsibility as a process flow only if risk-register evidence is present: "
            "risk identification/register, classification by category/time horizon/rating, monitoring frequency, "
            "scenario links, mitigation actions and ERM integration. Do not invent formal escalation thresholds."
        ),
    }


def extract_governance_evidence(payload: dict) -> dict:
    gov_records = payload.get("governance", [])
    board_minutes = payload.get("board_minutes", [])
    bank = payload.get("bank", {})
    reporting_kpis = payload.get("reporting_kpis", {})
    metadata = payload.get("metadata", {})

    reporting_year = int(metadata.get("reporting_year", 2024))

    gov_by_year = {
        str(r["reporting_year"]): r
        for r in gov_records
        if isinstance(r, dict) and "reporting_year" in r
    }

    gov_trend = []
    for year in ["2022", "2023", "2024"]:
        if year in gov_by_year:
            g = gov_by_year[year]
            gov_trend.append({
                "year": int(year),
                "esg_committee_meetings": g.get("esg_committee_meetings_per_year"),
                "board_climate_expertise_pct": g.get("board_climate_expertise_pct"),
                "ceo_esg_compensation_pct": g.get("ceo_esg_compensation_pct"),
                "all_exec_climate_remuneration_pct": g.get("all_exec_climate_remuneration_pct"),
                "climate_on_board_agenda_pct": g.get("climate_on_board_agenda_pct"),
                "management_committee_name": g.get("management_committee_name"),
                "board_full_meeting_frequency": g.get("board_full_meeting_frequency"),
            })

    PRIORITY_TOPICS = [
        "esg report", "scenario analysis", "transition plan", "net-zero",
        "net zero", "target", "carbon credit", "physical risk", "green finance"
    ]

    minutes_2024 = [
        m for m in board_minutes
        if isinstance(m, dict)
        and _safe_int(m.get("reporting_year")) == reporting_year
        and m.get("decision_made_flag") is True
        and _is_present(m.get("decision_summary"))
        and _is_present(m.get("meeting_id"))
    ]

    # Group by decision text so the same decision is not treated as two separate
    # Board/committee decisions when it appears in multiple meeting records.
    grouped_decisions = {}
    for m in minutes_2024:
        key = re.sub(r"\s+", " ", str(m.get("decision_summary", "")).lower().strip())
        if key not in grouped_decisions:
            grouped_decisions[key] = {
                "decision": _normalise_text(m.get("decision_summary")),
                "dates": set(),
                "committees": set(),
                "committee_types": set(),
                "topics_discussed": set(),
                "meeting_ids": set(),
                "ifrs_evidence_paras": set(),
            }

        grouped_decisions[key]["dates"].add(m.get("meeting_date"))
        grouped_decisions[key]["committees"].add(m.get("committee_name"))
        grouped_decisions[key]["committee_types"].add(m.get("committee_type"))
        grouped_decisions[key]["meeting_ids"].add(m.get("meeting_id"))
        grouped_decisions[key]["ifrs_evidence_paras"].add(m.get("ifrs_s2_para_evidence"))
        topics = str(m.get("climate_topics_discussed") or "").split("|")
        grouped_decisions[key]["topics_discussed"].update(t for t in topics if _is_present(t))

    selected_decisions = []
    for item in grouped_decisions.values():
        dates = sorted(x for x in item["dates"] if _is_present(x))
        committees = sorted(x for x in item["committees"] if _is_present(x))
        topics = sorted(x for x in item["topics_discussed"] if _is_present(x))
        meeting_ids = sorted(x for x in item["meeting_ids"] if _is_present(x))
        decision = item["decision"]
        score = sum(1 for term in PRIORITY_TOPICS if term in decision.lower() or term in " ".join(topics).lower())
        selected_decisions.append({
            "primary_date": dates[0] if dates else None,
            "dates": dates,
            "committees": committees,
            "committee_types": sorted(x for x in item["committee_types"] if _is_present(x)),
            "topics_discussed": topics,
            "decision": decision,
            "ifrs_evidence_paras": sorted(x for x in item["ifrs_evidence_paras"] if _is_present(x)),
            "internal_refs": [f"[REF:{mid}]" for mid in meeting_ids],
            "decision_group_score": score,
        })

    selected_decisions = sorted(
        selected_decisions,
        key=lambda d: (-d.get("decision_group_score", 0), d.get("primary_date") or "")
    )[:6]

    gov_2024 = gov_by_year.get(str(reporting_year), {})

    # Conservative evidence-gap assessment.
    governance_instrument_fields = [
        "committee_charter", "committee_terms_of_reference", "board_mandate",
        "esg_committee_mandate", "formal_climate_mandate",
        "governance_policy_reference", "committee_charter_climate_mandate",
    ]
    formal_mandate_available = any(_is_present(gov_2024.get(f)) for f in governance_instrument_fields)

    tradeoff_terms = [
        "tradeoff", "trade-off", "capital allocation", "profitability",
        "implementation cost", "risk appetite", "competing priority", "competing priorities"
    ]
    tradeoff_decisions = [
        m for m in minutes_2024
        if any(
            term in str(m.get("decision_summary", "")).lower()
            or term in str(m.get("climate_topics_discussed", "")).lower()
            for term in tradeoff_terms
        )
    ]
    board_tradeoff_evidence_available = len(tradeoff_decisions) > 0

    skills_process_fields = [
        "skills_matrix", "skills_assessment_process", "board_skills_review",
        "skills_adequacy_assessment", "director_training_frequency",
        "training_hours", "skills_gap_analysis",
    ]
    skills_adequacy_process_available = any(_is_present(gov_2024.get(f)) for f in skills_process_fields)

    assurance_scope = str(gov_2024.get("assurance_scope", ""))
    financed_emissions_2024 = reporting_kpis.get("financed_emissions_2024_tco2e")
    financed_in_scope = "financed" in assurance_scope.lower() or "scope 3" in assurance_scope.lower()

    assurance_scope_limitation = {
        "assurance_scope": assurance_scope,
        "external_assurance": gov_2024.get("external_assurance"),
        "provider": gov_2024.get("assurance_provider"),
        "standard": gov_2024.get("assurance_standard"),
        "financed_emissions_2024_tco2e": financed_emissions_2024,
        "financed_emissions_in_scope": financed_in_scope,
        "instruction": (
            "State that assurance covers only the stated scope. If the stated scope is Scope 1 and 2 emissions, "
            "do not imply financed emissions or other Scope 3 categories are assured. For a bank, explicitly clarify "
            "that financed emissions are outside the stated assurance scope based on available evidence."
        ),
    }

    management_evidence = extract_management_process_evidence(payload, year=reporting_year)

    return {
        "bank": {
            "name": bank.get("bank_name"),
            "bank_id": bank.get("bank_id"),
            "country": bank.get("country"),
            "total_assets_meur": bank.get("total_assets_meur"),
            "regulatory_regime": bank.get("regulatory_regime"),
        },
        "reporting_year": reporting_year,
        "comparative_years": metadata.get("comparative_years", [2022, 2023]),
        "payload_profile": {
            "source_payload": "governance",
            "top_level_keys": list(payload.keys()),
            "climate_risk_register_in_payload": "climate_risk_register" in payload,
            "board_minutes_count": len(board_minutes),
            "governance_record_count": len(gov_records),
        },
        "governance_2024": {
            "board_size": gov_2024.get("board_size"),
            "independent_directors_pct": gov_2024.get("independent_directors_pct"),
            "esg_committee_exists": gov_2024.get("esg_committee_exists"),
            "esg_committee_meetings_per_year": gov_2024.get("esg_committee_meetings_per_year"),
            "board_climate_expertise_pct": gov_2024.get("board_climate_expertise_pct"),
            "ceo_compensation_esg_linked": gov_2024.get("ceo_compensation_esg_linked"),
            "ceo_esg_compensation_pct": gov_2024.get("ceo_esg_compensation_pct"),
            "all_exec_climate_remuneration_pct": gov_2024.get("all_exec_climate_remuneration_pct"),
            "climate_risk_reporting_to_board": gov_2024.get("climate_risk_reporting_to_board"),
            "climate_on_board_agenda_pct": gov_2024.get("climate_on_board_agenda_pct"),
            "board_full_meeting_frequency": gov_2024.get("board_full_meeting_frequency"),
            "management_committee_name": gov_2024.get("management_committee_name"),
            "erm_integration_flag": gov_2024.get("erm_integration_flag"),
            "skills_development_programme": gov_2024.get("skills_development_programme"),
            "major_transactions_climate_check": gov_2024.get("major_transactions_climate_check"),
            "external_assurance": gov_2024.get("external_assurance"),
            "assurance_provider": gov_2024.get("assurance_provider"),
            "assurance_scope": gov_2024.get("assurance_scope"),
            "assurance_standard": gov_2024.get("assurance_standard"),
            "tcfd_aligned": gov_2024.get("tcfd_aligned"),
            "ifrs_s2_aligned": gov_2024.get("ifrs_s2_aligned"),
        },
        "governance_trend": gov_trend,
        "management_process_evidence": management_evidence,
        "board_decisions_2024": selected_decisions,
        "assurance_context": {
            "financed_emissions_2024_tco2e": financed_emissions_2024,
            "assurance_scope": assurance_scope,
            "financed_emissions_in_scope": financed_in_scope,
        },
        "strict_governance_evidence": {
            "formal_governance_mandate_available": formal_mandate_available,
            "formal_governance_mandate_instruction": (
                "Do not claim the ESG & Sustainability Committee has a formal climate mandate unless charter "
                "or terms-of-reference evidence is provided. If no formal instrument is available, say that "
                "available documentation evidences committee activity and meeting frequency but does not include "
                "the committee charter or terms of reference."
            ),
            "board_tradeoff_evidence_available": board_tradeoff_evidence_available,
            "tradeoff_decisions": tradeoff_decisions[:3],
            "board_tradeoff_instruction": (
                "Discuss board trade-offs only if explicit trade-off evidence exists. If not, state that the board "
                "decision evidence identifies climate-related decisions but does not describe specific trade-offs "
                "such as profitability, capital allocation, implementation cost, risk appetite or competing strategic priorities."
            ),
            "skills_adequacy_process_available": skills_adequacy_process_available,
            "skills_adequacy_instruction": (
                "Use the board climate expertise percentage and skills development programme as outcome/activity evidence. "
                "Do not invent a formal skills adequacy assessment process."
            ),
            "assurance_scope_limitation": assurance_scope_limitation,
        },
        "interpretation_notes": {
            "climate_on_board_agenda_pct": (
                "This figure represents the percentage of board meetings during the year where climate-related topics "
                "appeared on the agenda. It does not mean percentage of agenda time devoted to climate."
            ),
            "management_committee_names": (
                "Committee names are recorded by year only. The evidence does not prove that one committee evolved into, "
                "replaced, or was renamed as another. State the 2024 committee name and, if comparative names are used, "
                "present them neutrally."
            ),
            "major_transactions_climate_check": (
                "A true value indicates evidence of climate checks for major transactions; it does not prove a formal mandatory policy."
            ),
            "board_decision_traceability": (
                "Grouped decisions include internal refs for audit traceability. Do not print these refs in the final report."
            ),
            "avoid_duplication": (
                "Do not list the same board decisions twice. Board oversight should summarise decision governance; "
                "the detailed dated list belongs only in the Board and committee decisions subsection."
            ),
            "carbon_credit_boundary": (
                "Carbon credit procurement may be described only as a specific 2024 budget approval decision unless recurring topic evidence is available."
            ),
            "alignment_boundary": (
                "TCFD/IFRS S2 alignment flags show source-data alignment status only. Do not claim governance oversight supports or ensures alignment unless an alignment control process is evidenced."
            ),
        },
    }


evidence = extract_governance_evidence(governance_payload)

print(f"Evidence extracted for: {evidence['bank']['name']}")
print(f"Governance payload risk register present: {evidence['payload_profile']['climate_risk_register_in_payload']}")
print(f"Board decisions selected: {len(evidence['board_decisions_2024'])}")
print(f"Trend years: {[t['year'] for t in evidence['governance_trend']]}")
print(f"Management evidence risk register available: {evidence['management_process_evidence'].get('risk_register_available')}")
print("Strict governance evidence flags:")
for k, v in evidence["strict_governance_evidence"].items():
    if isinstance(v, bool):
        print(f"- {k}: {v}")
print("Grouped decisions:")
for d in evidence["board_decisions_2024"]:
    print(f"- {d.get('primary_date')} | {', '.join(d.get('committees', []))} | {d.get('decision')}")

## 4. Build compact Governance evidence


In [ ]:
# ── GOVERNANCE EVIDENCE AVAILABILITY + SAVING ────────────────
# The raw Governance payload remains the source of truth.
# The compact evidence is the agent-ready input used by Writer/Judge/Reviser.

def is_scope1_scope2_only(scope: str | None) -> bool:
    if not scope:
        return False
    s = str(scope).lower()
    s = re.sub(r"\s+", " ", s)
    has_scope1 = bool(re.search(r"\bscope\s*1\b|\bscope1\b", s))
    has_scope2 = bool(
        re.search(r"\bscope\s*2\b|\bscope2\b", s)
        or re.search(r"\bscope\s*1\s*(and|&)\s*2\b", s)
        or re.search(r"\bscope\s*1\s*(and|&)\s*scope\s*2\b", s)
    )
    has_scope3_or_financed = bool(
        re.search(r"\bscope\s*3\b|\bscope3\b|financed emissions|category 15|cat\.?\s*15", s)
    )
    return has_scope1 and has_scope2 and not has_scope3_or_financed

def build_governance_availability_profile(evidence: dict) -> dict:
    gov = evidence.get("governance_2024", {})
    trend = evidence.get("governance_trend", [])
    management = evidence.get("management_process_evidence", {})
    strict = evidence.get("strict_governance_evidence", {})
    decisions = evidence.get("board_decisions_2024", [])

    def present(value) -> bool:
        return value is not None and str(value).strip().lower() not in {
            "", "none", "null", "nan"
        }

    trend_metrics = {
        "esg_committee_meetings": [
            row.get("esg_committee_meetings") for row in trend
            if present(row.get("esg_committee_meetings"))
        ],
        "board_climate_expertise_pct": [
            row.get("board_climate_expertise_pct") for row in trend
            if present(row.get("board_climate_expertise_pct"))
        ],
        "ceo_esg_compensation_pct": [
            row.get("ceo_esg_compensation_pct") for row in trend
            if present(row.get("ceo_esg_compensation_pct"))
        ],
        "all_exec_climate_remuneration_pct": [
            row.get("all_exec_climate_remuneration_pct") for row in trend
            if present(row.get("all_exec_climate_remuneration_pct"))
        ],
        "climate_on_board_agenda_pct": [
            row.get("climate_on_board_agenda_pct") for row in trend
            if present(row.get("climate_on_board_agenda_pct"))
        ],
        "board_full_meeting_frequency": [
            row.get("board_full_meeting_frequency") for row in trend
            if present(row.get("board_full_meeting_frequency"))
        ],
    }

    assurance_scope = str(gov.get("assurance_scope") or "").lower()
    financed_emissions = evidence.get("assurance_context", {}).get(
        "financed_emissions_2024_tco2e"
    )

    risk_register_available = bool(management.get("risk_register_available"))
    governance_controls_available = bool(management.get("governance_controls_available"))

    return {
        "board_core_metrics_available": all(
            present(gov.get(field))
            for field in [
                "board_size",
                "independent_directors_pct",
                "esg_committee_meetings_per_year",
                "climate_risk_reporting_to_board",
                "climate_on_board_agenda_pct",
            ]
        ),
        "trend_metrics_available": {
            name: len(values) >= 2
            for name, values in trend_metrics.items()
        },
        "selected_decision_count": len(decisions),
        "board_decisions_available": len(decisions) > 0,
        "formal_governance_mandate_available": bool(
            strict.get("formal_governance_mandate_available")
        ),
        "board_tradeoff_evidence_available": bool(
            strict.get("board_tradeoff_evidence_available")
        ),
        "management_process_available": risk_register_available or governance_controls_available,
        "management_risk_register_available": risk_register_available,
        "management_governance_controls_available": governance_controls_available,
        "formal_escalation_thresholds_available": bool(
            management.get("formal_escalation_thresholds_available", False)
        ),
        "skills_outcome_metrics_available": present(
            gov.get("board_climate_expertise_pct")
        ),
        "skills_development_programme_available": bool(
            gov.get("skills_development_programme")
        ),
        "skills_adequacy_process_available": bool(
            strict.get("skills_adequacy_process_available")
        ),
        "remuneration_evidence_available": (
            present(gov.get("ceo_esg_compensation_pct"))
            and present(gov.get("all_exec_climate_remuneration_pct"))
        ),
        "assurance_evidence_available": all(
            present(gov.get(field))
            for field in [
                "external_assurance",
                "assurance_provider",
                "assurance_scope",
                "assurance_standard",
            ]
        ),
        "assurance_scope_is_scope1_scope2_only": is_scope1_scope2_only(gov.get("assurance_scope")),
        "financed_emissions_available_for_scope_context": present(
            financed_emissions
        ),
        "payload_boundary": {
            "governance_payload_has_climate_risk_register": evidence.get("payload_profile", {}).get("climate_risk_register_in_payload"),
            "governance_payload_tables": evidence.get("payload_profile", {}).get("top_level_keys", []),
        },
        "writer_policy": {
            "use_available_evidence": (
                "Use every material governance evidence item that is available and relevant."
            ),
            "handle_unavailable_evidence": (
                "When a material governance requirement is not supported by the available evidence, state the boundary once in the relevant subsection. Do not invent the missing process or control."
            ),
            "management_process_boundary": (
                "If climate_risk_register is unavailable, describe only the management committee, board reporting frequency, ERM integration flag and major-transaction climate check. If it is available, describe the risk-register process but do not invent formal escalation thresholds."
            ),
            "wording": (
                "Use 'available evidence', 'available documentation', or 'source data' in final disclosure; do not use the word 'payload'."
            ),
        },
    }


def add_governance_traceability(evidence: dict, source_payload_path: Path) -> dict:
    evidence = dict(evidence)
    evidence["availability_profile"] = build_governance_availability_profile(evidence)
    source_tables = [
        "bank",
        "governance",
        "board_minutes",
        "reporting_kpis",
    ]
    if evidence.get("payload_profile", {}).get("climate_risk_register_in_payload"):
        source_tables.append("climate_risk_register")

    evidence["source_traceability"] = {
        "source_payload_path": str(source_payload_path),
        "source_tables": source_tables,
        "selected_board_decision_refs": [
            item.get("internal_ref")
            for item in evidence.get("board_decisions_2024", [])
            if item.get("internal_ref")
        ],
        "management_risk_refs": [
            item.get("risk_id")
            for item in evidence.get("management_process_evidence", {}).get(
                "material_risk_examples", []
            )
            if item.get("risk_id")
        ],
    }
    return evidence


evidence = add_governance_traceability(evidence, GOVERNANCE_PAYLOAD_PATH)

raw_governance_payload = {
    key: governance_payload.get(key)
    for key in [
        "metadata",
        "bank",
        "governance",
        "board_minutes",
        "climate_risk_register",
        "reporting_kpis",
    ]
    if key in governance_payload
}

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

RAW_GOVERNANCE_PATH = output_dir / "payload_BANK01_governance_raw.json"
COMPACT_GOVERNANCE_PATH = output_dir / "compact_governance_evidence_BANK01.json"

with open(RAW_GOVERNANCE_PATH, "w", encoding="utf-8") as f:
    json.dump(raw_governance_payload, f, indent=2, ensure_ascii=False)

with open(COMPACT_GOVERNANCE_PATH, "w", encoding="utf-8") as f:
    json.dump(evidence, f, indent=2, ensure_ascii=False)

print("Governance evidence prepared")
print(f"- Raw Governance payload: {RAW_GOVERNANCE_PATH}")
print(f"- Compact Governance evidence: {COMPACT_GOVERNANCE_PATH}")
print("- Availability profile:")
print(json.dumps(evidence["availability_profile"], indent=2, ensure_ascii=False))

## 5. Add traceability and save Governance evidence


In [ ]:
# ── STATE DEFINITION ─────────────────────────────────────────
class GovernanceState(TypedDict):
    bank_name:        str
    evidence:         dict
    draft:            str
    judge_result:     dict
    revision_count:   int
    max_revisions:    int
    status:           Literal["drafting", "judging", "revising", "approved", "failed"]
    final_section:    str
    token_usage:      dict

## 6. Governance state definition


In [ ]:
# ── GOVERNANCE REQUIREMENTS ─────────────────────────────────
# IFRS S1/S2 references are used internally only. Final text must not show paragraph tags.
# Structure is adapted to a dedicated banking IFRS S1/S2 disclosure report style.

IFRS_GOVERNANCE_REQUIREMENTS = """
STRICT GOVERNANCE DISCLOSURE REQUIREMENTS FOR THIS SECTION:

Final section title and headings must be exactly:
### Governance
#### Overview
#### The role of the Board of Directors
#### Board committees and climate-related oversight
#### Management responsibility for climate-related risks and opportunities
#### Skills, competencies and remuneration
#### Governance decisions, controls and evidence boundaries

Specific wording controls:
- Keep semi-annual Board reporting separate from individual Board/committee decision evidence. Do not imply that net-zero target revisions, transition-plan updates or scenario-methodology approvals were part of the semi-annual reporting pack unless that linkage is explicitly evidenced.
- Use evidence-anchored risk-register wording such as "uses a climate risk register" or "documents climate-related risks in a climate risk register". Avoid "maintains a dedicated climate risk register" because it implies an ownership/control process not directly evidenced.
- Avoid broad wording such as "supporting processes embedded in risk management and control frameworks". Prefer specific evidenced wording such as "supporting processes reflected in ERM integration flags, climate risk reporting and climate checks for major transactions".

Presentation style:
- Write like a dedicated bank IFRS S1/S2 sustainability disclosure report, not like a checklist answer.
- Start with an overview of how climate-related governance is organised across the Board, committees, management and control functions.
- Use smooth report-style wording.
- Do not overuse the phrase "available evidence" in the main narrative. Use "source data", "available documentation" or "documentation reviewed" mainly for limitations and evidence boundaries.

Internal IFRS S2 Governance coverage map:
1. Governance mandate / terms of reference / role descriptions:
   - If formal mandate evidence is available, describe it.
   - If unavailable, state that available documentation evidences governance activity and meeting frequency but does not include a separate climate-specific charter, terms of reference or formal mandate.
2. Board oversight:
   - Use board size, independent-director percentage, full Board meeting frequency and climate agenda frequency.
   - Use the three-year trend for full Board meetings and climate-on-board-agenda frequency.
   - Explain that Board oversight is evidenced through agenda coverage, reporting frequency, Board/committee decisions and governance KPIs.
3. Board committees and climate-related oversight:
   - Use ESG Committee meeting counts and trends.
   - Use grouped 2024 Board/committee decisions without duplicating a long dated list in earlier subsections.
   - Use target oversight and transition-plan/scenario-methodology approvals only where evidenced.
4. Skills and competencies:
   - Use board climate expertise percentage and the skills development programme.
   - If no formal skills adequacy assessment is available, state this boundary.
5. Remuneration:
   - Use CEO ESG-linked compensation percentage and all-executive climate-linked remuneration percentage.
   - Include the three-year trend for remuneration metrics where available.
6. Trade-offs:
   - Use only explicit trade-off evidence.
   - If unavailable, state that documented decisions do not describe quantified or specific Board-level trade-offs.
7. Management responsibility:
   - Identify the management committee responsible for climate-related risks and opportunities.
   - Use reporting frequency to the Board, ERM integration flag, major-transaction climate check and climate risk register evidence where available.
8. Management controls and procedures:
   - If risk register evidence is available, describe the process at a summary level: risk identification/register, categorisation, monitoring frequency, scenario links, mitigation actions and ERM integration.
   - If unavailable, limit the description to the management committee, Board reporting, ERM integration and major-transaction climate checks.
   - In both cases, do not invent formal escalation thresholds.
9. Assurance and controls:
   - Describe assurance only for the stated scope.
   - If assurance scope is Scope 1 and Scope 2 emissions, state that financed emissions / Scope 3 are outside the stated assurance scope when financed-emissions context is available.
10. Evidence boundaries:
   - The final subsection must include concise boundaries for unavailable formal mandate/charter evidence, Board trade-off evidence, skills adequacy assessment, escalation thresholds, and assurance scope where applicable.

Data-aware coverage requirements:
1. Use the Governance evidence as the source of governance facts.
2. If climate_risk_register is present, Management responsibility must describe the evidenced risk-register process.
3. If climate_risk_register is absent, Management responsibility must be limited to management committee name, climate risk reporting frequency, ERM integration flag and major-transaction climate check flag.
4. Do not invent formal committee charters, terms of reference, formal governance mandates, formal escalation thresholds, Board trade-off analysis or formal skills adequacy assessment.
5. Use year-on-year trends for available metrics: ESG committee meetings 5→6→7; board climate expertise 33.5%→35.5%→37.5%; CEO ESG-linked compensation 6.3%→7.8%→9.3%; all-executive climate-linked remuneration 8.7%→15.3%→15.1%; climate agenda frequency 69.8%→73.1%→72.6%; full Board meetings 4→7→8.
6. Carbon credit procurement may be described only as a specific 2024 budget approval decision unless recurring topic evidence is available.
7. Alignment flags may be disclosed only as source-data flags. Do not claim governance oversight supports or ensures TCFD/IFRS S2 alignment unless an alignment control process is evidenced.
8. Do not include visible IFRS paragraph references, meeting IDs, internal refs or bracketed IFRS tags in the final output.
9. Do not use the technical word "payload" in the final report.
""".strip()

print("Governance requirements ready — Emirates-style structure")

## 7. Governance requirements — Emirates-style structure


In [ ]:
# ── GOVERNANCE WRITER PROMPT ─────────────────────────────────
WRITER_SYSTEM = """
You are a senior sustainability reporting specialist writing the Governance section of a dedicated IFRS S1/S2-style climate disclosure report for a commercial bank.

Write in a formal, third-person, publication-ready style similar to a bank IFRS S1/S2 disclosure report.

Evidence rules:
- Use only the compact Governance evidence supplied by the user prompt.
- Do not invent missing governance policies, committee charters, trade-offs, escalation thresholds, risk-register workflows, assurance coverage or skills assessment processes.
- When evidence is missing, state the limitation in report-style language using "source data", "available documentation", "documentation reviewed" or "available evidence".
- Do not use the word "payload" in the final section.
- Do not include IFRS paragraph references, meeting IDs, internal references or bracketed evidence tags.
- Do not use markdown tables.

Style rules:
- Write like a report section, not like an audit checklist.
Precision wording rules:
- Do not write that Board reporting is "supplemented by" net-zero, transition-plan or scenario-methodology decisions. State semi-annual Board reporting and separate 2024 decision evidence as two distinct channels.
- Do not use "maintains a dedicated climate risk register". Use: "uses a climate risk register that, in 2024, recorded 8 climate-related risks" when risk-register evidence is available.
- Do not use "supporting processes embedded". Use: "supporting processes reflected in ERM integration flags, climate risk reporting and climate checks for major transactions".
- Do not state that decision items are part of a reporting pack unless the evidence explicitly says so.
- The section must open with an "Overview" explaining the governance model across Board, committees, management and controls.
- Do not overuse "available evidence" in every paragraph. Use it mainly when describing limitations.
- Use smooth banking-report language: "oversight is reflected in", "the governance structure includes", "source data indicates", "documentation reviewed does not specify".
- Avoid strong interpretive verbs such as "demonstrates", "ensures", "supports", "confirms" or "drives" unless the evidence directly proves the mechanism.
- Prefer "indicates", "is evidenced by", "is reflected in" or "source data shows".
- Do not claim that governance oversight supports or ensures TCFD/IFRS S2 alignment; state only that source data flags alignment if relevant.

Output rules:
- Return only the complete Governance section.
- Keep exactly the six required subsections.
""".strip()


def build_writer_prompt(evidence: dict, judge_feedback: str = None) -> str:
    profile = evidence.get("availability_profile", {})
    gov = evidence.get("governance_2024", {})
    management = evidence.get("management_process_evidence", {})
    strict = evidence.get("strict_governance_evidence", {})

    instructions = []

    instructions.append(
        "- Use board size, independent-director percentage, full-board meeting count, climate agenda percentage, committee meeting count and board reporting frequency."
    )
    instructions.append(
        "- Include the available three-year trend narrative for ESG committee meetings, board climate expertise, CEO ESG-linked pay, all-executive climate-linked pay, climate agenda frequency and full Board meeting frequency."
    )
    instructions.append(
        "- Use the Emirates-style adapted structure: Overview; role of the Board; Board committees; Management responsibility; Skills/competencies/remuneration; Decisions/controls/boundaries."
    )
    instructions.append(
        "- Keep the detailed dated decisions in the final Governance decisions, controls and evidence boundaries subsection. Do not duplicate the full list earlier."
    )
    instructions.append(
        "- Keep semi-annual Board reporting separate from 2024 Board/committee decisions. Do not imply that decision items formed part of the semi-annual reporting pack."
    )
    instructions.append(
        "- Use evidence-anchored wording for the risk register: 'uses a climate risk register that, in 2024, recorded 8 climate-related risks'. Avoid 'maintains a dedicated climate risk register'."
    )
    instructions.append(
        "- Replace broad process language with evidenced mechanisms: ERM integration flags, climate risk reporting and climate checks for major transactions."
    )

    if profile.get("formal_governance_mandate_available"):
        instructions.append("- Describe the formal governance mandate using the supplied direct evidence.")
    else:
        instructions.append(
            "- Committee and Board activity are evidenced, but no separate climate-specific committee charter/terms of reference/formal mandate is available. State this boundary once, preferably in the final boundaries subsection."
        )

    if profile.get("board_tradeoff_evidence_available"):
        instructions.append("- Describe only the documented Board trade-offs included in evidence.")
    else:
        instructions.append(
            "- Board decisions are evidenced, but quantified or specific Board-level trade-offs are not documented. State this boundary once without inventing trade-offs."
        )

    if profile.get("management_risk_register_available"):
        instructions.append(
            "- Management responsibility must describe the available risk-register process: eight 2024 risks, risk categories, risk ratings, time horizons, monitoring frequencies, scenario links, mitigation actions and ERM integration count."
        )
    elif profile.get("management_governance_controls_available"):
        instructions.append(
            "- Management responsibility must NOT describe a risk-register workflow. Use only the Climate Risk Management Committee, semi-annual Board reporting, ERM integration flag and major-transaction climate check."
        )
    else:
        instructions.append(
            "- Management process evidence is not available. State the boundary without inventing a process."
        )

    if profile.get("formal_escalation_thresholds_available"):
        instructions.append("- Describe formal escalation thresholds/routes exactly as evidenced.")
    else:
        instructions.append(
            "- Formal escalation thresholds are not evidenced. Use boundary wording such as: documentation reviewed does not evidence formal escalation thresholds or trigger-based escalation mechanics."
        )

    if profile.get("skills_adequacy_process_available"):
        instructions.append("- Describe the formal Board skills adequacy assessment process from evidence.")
    else:
        instructions.append(
            "- Use Board climate expertise percentage and skills development programme only. State that no formal Board skills adequacy assessment process is documented. Do not invent detailed training topics."
        )

    if profile.get("remuneration_evidence_available"):
        instructions.append(
            "- Use both CEO ESG-linked remuneration percentage and all-executive climate-linked remuneration percentage, including available comparative trend values."
        )

    if profile.get("assurance_evidence_available"):
        instructions.append(
            "- Describe assurance using exact provider, level, standard and scope."
        )
        if profile.get("assurance_scope_is_scope1_scope2_only"):
            instructions.append(
                "- Clarify that assurance covers Scope 1 and Scope 2 emissions only, and financed emissions / Scope 3 are outside the stated assurance scope."
            )

    section_blueprint = """
SECTION BLUEPRINT:
### Governance

#### Overview
Summarise the climate governance architecture: Board oversight, Board/committee activity, management responsibility, risk-register/ERM linkage if available, skills/remuneration and controls. Keep it concise.

#### The role of the Board of Directors
Use Board size, independence, Board meeting frequency, climate agenda frequency, Board reporting frequency and Board oversight facts. Include the relevant trend values naturally.

#### Board committees and climate-related oversight
Use ESG Committee meeting trend, committee oversight facts and grouped Board/committee decision themes. Do not create formal committee charters or terms of reference if unavailable.

#### Management responsibility for climate-related risks and opportunities
Use management committee name, reporting frequency to the Board, ERM integration, major-transaction climate check and climate risk register process if available. Use 'uses a climate risk register that, in 2024, recorded 8 climate-related risks' rather than 'maintains a dedicated climate risk register'.

#### Skills, competencies and remuneration
Use Board climate expertise trend, skills development evidence, CEO ESG-linked remuneration and all-executive climate-linked remuneration. State skills adequacy boundary if unavailable.

#### Governance decisions, controls and evidence boundaries
Include dated/grouped 2024 decisions, assurance scope, governance controls and concise evidence boundaries for missing formal mandate, trade-offs, skills adequacy assessment and escalation thresholds.
""".strip()

    feedback_block = ""
    if judge_feedback:
        feedback_block = f"""
JUDGE FEEDBACK TO ADDRESS:
{judge_feedback}

REVISION RULES:
- Fix the judge's valid issues using available evidence.
- Preserve correct evidence-boundary statements.
- Never invent unavailable information.
""".strip()

    return f"""
{IFRS_GOVERNANCE_REQUIREMENTS}

BANK:
{evidence['bank']['name']} ({evidence['bank']['country']})
REPORTING YEAR: {evidence['reporting_year']}
COMPARATIVE YEARS: {evidence['comparative_years']}

AVAILABLE-EVIDENCE PROFILE:
{json.dumps(profile, indent=2, ensure_ascii=False)}

DATA-AWARE WRITING INSTRUCTIONS:
{chr(10).join(instructions)}

{section_blueprint}

COMPACT GOVERNANCE EVIDENCE — USE ONLY THIS DATA:
{json.dumps(evidence, indent=2, ensure_ascii=False)}

IMPORTANT DATA BOUNDARIES:
- Governance evidence tables available: {evidence.get('payload_profile', {}).get('top_level_keys')}
- Climate risk register present in Governance evidence: {evidence.get('payload_profile', {}).get('climate_risk_register_in_payload')}
- Management process instruction: {management.get('process_flow_instruction')}
- Formal mandate instruction: {strict.get('formal_governance_mandate_instruction')}
- Board trade-off instruction: {strict.get('board_tradeoff_instruction')}
- Skills instruction: {strict.get('skills_adequacy_instruction')}
- Assurance instruction: {strict.get('assurance_scope_limitation', {}).get('instruction')}

GENERAL WRITING RULES:
- Do not duplicate the detailed Board-decision list before the final decisions/boundaries subsection.
- Keep semi-annual climate risk reporting and separate Board/committee decision evidence distinct.
- Do not imply that net-zero target revisions, transition-plan updates or climate scenario methodology approvals were part of the semi-annual Board reporting pack unless explicitly evidenced.
- Do not use the phrase 'maintains a dedicated climate risk register'.
- Do not use the phrase 'supporting processes embedded'.
- Use dates and decision descriptions mainly in the final Governance decisions, controls and evidence boundaries subsection.
- Interpret climate_on_board_agenda_pct as the percentage of Board meetings where climate appeared on the agenda.
- A true major_transactions_climate_check flag indicates evidence of climate checks; it does not prove a formal mandatory policy.
- Do not infer that committee names from 2022, 2023 and 2024 represent the same renamed committee.
- Do not describe carbon credit procurement as recurring governance unless topic evidence explicitly supports that. If it appears only as a decision title, mention it only as a 2024 decision.
- When alignment flags such as TCFD-aligned or IFRS S2-aligned are present, state only that source data flags the disclosure as aligned; do not claim governance oversight supports or ensures alignment.

{feedback_block}

Write the complete Governance section now.
Return only the final Governance section.
""".strip()

## 8. Governance writer prompt


In [ ]:
# ── GOVERNANCE JUDGE PROMPT ─────────────────────────────────
JUDGE_SYSTEM = """
You are a strict sustainability-reporting judge for an IFRS S1/S2-aligned bank climate disclosure.

You evaluate whether the Governance section is:
- supported by the compact evidence;
- aligned with the Governance disclosure checklist;
- structured like a dedicated bank IFRS S1/S2 report section rather than a checklist answer;
- transparent about missing evidence;
- free from hallucinated policies, processes, trade-offs, assurance coverage and unsupported governance claims.

Return valid JSON only.
All checklist values must be JSON booleans true or false, not strings such as "true" or "false".
""".strip()


def build_judge_prompt(draft: str, evidence: dict, deterministic_checks: dict | None = None) -> str:
    profile = evidence.get("availability_profile", {})
    deterministic_checks = deterministic_checks or {}

    return f"""
Evaluate the Governance draft against the compact evidence, availability profile and deterministic pre-checks.

DRAFT:
{draft}

AVAILABLE-EVIDENCE PROFILE:
{json.dumps(profile, indent=2, ensure_ascii=False)}

DETERMINISTIC PRE-CHECKS:
{json.dumps(deterministic_checks, indent=2, ensure_ascii=False)}

COMPACT GOVERNANCE EVIDENCE:
{json.dumps(evidence, indent=2, ensure_ascii=False)}

EVALUATION PRINCIPLES:
1. Penalise any claim that contradicts evidence or invents unavailable governance information.
2. Penalise omission when material evidence is available but not used.
3. Do not demand unavailable evidence. Correctly disclosed evidence boundaries are acceptable, but lower completeness.
4. Verify that the section follows the adapted bank IFRS S1/S2 style structure:
   Overview; role of the Board; Board committees; management responsibility; skills/competencies/remuneration; decisions/controls/boundaries.
5. Distinguish committee activity evidence from formal charter/mandate evidence.
6. Distinguish governance-level management controls from risk-register process evidence.
7. If climate_risk_register is available, the draft should describe the management risk-register process at a reasonable summary level: risk identification/register, risk categories, time horizons, ratings, monitoring frequencies, scenario links, mitigation actions and ERM integration.
8. If climate_risk_register is absent, the draft must not describe risk counts, risk categories, scenario links, monitoring frequencies by risk, mitigation actions or a risk-register workflow.
9. If formal mandate, Board trade-offs, formal escalation thresholds or skills adequacy assessment are unavailable, the draft must not invent them.
10. Verify exact figures and trends: 10 Board members; 68.5% independence; 72.6% 2024 climate agenda frequency; ESG Committee meetings 5→6→7; Board climate expertise 33.5%→35.5%→37.5%; CEO ESG remuneration 6.3%→7.8%→9.3%; all-executive climate remuneration 8.7%→15.3%→15.1%.
11. Verify that assurance is limited to Scope 1 and Scope 2 emissions when that is the stated scope and does not imply financed emissions assurance.
12. Verify no visible IFRS paragraph references, meeting IDs or internal refs appear in the final section.
13. Verify decisions are not unnecessarily duplicated across multiple subsections.
14. Do not penalise accurate limitation statements such as "available documentation does not evidence formal escalation thresholds".
15. Alignment flags should be described only as source-data flags, not as proof that governance oversight ensures alignment.
16. Do not link semi-annual reporting to specific decision items unless the evidence explicitly states that the decisions were part of the reporting pack.
17. Prefer evidence-anchored risk-register wording such as 'uses a climate risk register'; penalise stronger operational-control wording such as 'maintains a dedicated climate risk register' when ownership/control maintenance is not evidenced.

SCORING:
- 9–10: Directly evidenced, materially complete, report-style structure, no unsupported claims and no material evidence boundary.
- 8: Strong, report-ready, with only one material boundary correctly disclosed.
- 7: Usable, with multiple correctly disclosed boundaries or a somewhat checklist-like style.
- 6: Revision required because available evidence is omitted, unsupported wording remains, or the new structure is not followed.
- 5 or below: Major evidence failure, hallucination, contradiction or missing core subsection.

Return valid JSON only. All checklist values must be booleans true/false, not strings.
Keep arrays concise:
{{
  "overall_score": <integer 1-10>,
  "evidence_support_score": <integer 1-10>,
  "ifrs_alignment_score": <integer 1-10>,
  "specificity_score": <integer 1-10>,
  "hallucination_risk": "<low|medium|high>",
  "approval_status": "<approved|approved_with_limitations|revision_required|rejected>",
  "approved": <true if approved or approved_with_limitations, otherwise false>,
  "checklist": {{
    "required_structure_present": <true/false>,
    "report_style_structure_present": <true/false>,
    "board_metrics_used_correctly": <true/false>,
    "trend_metrics_used_correctly": <true/false>,
    "board_decisions_used_correctly": <true/false>,
    "formal_mandate_handled_according_to_availability": <true/false>,
    "board_tradeoffs_handled_according_to_availability": <true/false>,
    "management_evidence_handled_according_to_availability": <true/false>,
    "risk_register_used_if_available_or_not_invented_if_absent": <true/false>,
    "escalation_handled_according_to_availability": <true/false>,
    "skills_handled_according_to_availability": <true/false>,
    "remuneration_evidence_used_correctly": <true/false>,
    "assurance_scope_used_correctly": <true/false>,
    "no_unsupported_committee_evolution": <true/false>,
    "no_duplicate_decisions": <true/false>,
    "no_visible_ifrs_refs_or_internal_refs": <true/false>,
    "no_unsupported_strong_claims": <true/false>
  }},
  "available_evidence_omitted": [<specific available evidence omitted>],
  "unsupported_claims": [<specific unsupported claims>],
  "correctly_disclosed_evidence_boundaries": [<accurate boundary statements>],
  "main_issues": [<specific issues>],
  "required_fixes": [<actionable evidence-aware fixes>]
}}
""".strip()

## 9. Governance judge prompt


In [ ]:
# ── GOVERNANCE EVALUATION MODE ───────────────────────────────
# Data-aware deterministic pre-checks are now active.
# They do not directly approve/reject the section or cap the score.
# Instead, their findings are passed to the GPT-5.2 Judge so the Judge can evaluate
# the draft against the actual available Governance data.

print("Governance evaluation mode: GPT-5.2 judge + data-aware deterministic pre-checks")

## 10. Governance evaluation mode


In [ ]:
# ── GOVERNANCE LANGGRAPH NODES ───────────────────────────────

def normalize_json_booleans(obj):
    """Normalize LLM JSON outputs where booleans may be returned as strings."""
    if isinstance(obj, dict):
        return {k: normalize_json_booleans(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [normalize_json_booleans(v) for v in obj]
    if isinstance(obj, str):
        if obj.strip().lower() == "true":
            return True
        if obj.strip().lower() == "false":
            return False
    return obj

def _contains_any(text: str, phrases: list[str]) -> bool:
    text_l = text.lower()
    return any(p.lower() in text_l for p in phrases)


def run_governance_deterministic_checks(draft: str, evidence: dict) -> dict:
    """Data-aware deterministic checks passed to the Judge as evidence-aware signals."""
    text = draft or ""
    text_l = text.lower()
    profile = evidence.get("availability_profile", {})
    gov = evidence.get("governance_2024", {})

    required_headings = [
        "#### Overview",
        "#### The role of the Board of Directors",
        "#### Board committees and climate-related oversight",
        "#### Management responsibility for climate-related risks and opportunities",
        "#### Skills, competencies and remuneration",
        "#### Governance decisions, controls and evidence boundaries",
    ]

    missing_headings = [h for h in required_headings if h not in text]

    warnings = []
    failures = []

    if missing_headings:
        failures.append({"check": "missing_required_headings", "details": missing_headings})


    # Specific wording checks to prevent minor overreach found during evaluation.
    if "supplemented by specific decision items" in text_l:
        warnings.append({
            "check": "semi_annual_reporting_decision_linkage_overstated",
            "details": "Keep semi-annual Board reporting separate from individual Board/committee decision evidence."
        })

    if "maintains a dedicated climate risk register" in text_l:
        warnings.append({
            "check": "risk_register_control_wording_overstated",
            "details": "Use 'uses a climate risk register' or 'documents climate-related risks in a climate risk register' instead."
        })

    if "supporting processes embedded" in text_l:
        warnings.append({
            "check": "supporting_processes_embedding_overstated",
            "details": "Use specific evidenced mechanisms instead of broad embedded-process language."
        })

    # Report-style structure checks.
    if "#### Overview" in text:
        overview_text = text.split("#### Overview", 1)[1].split("#### The role of the Board of Directors", 1)[0].lower()
        if len(overview_text.split()) < 50:
            warnings.append({
                "check": "overview_may_be_too_short",
                "details": "Overview is present but may be too short to explain the governance model."
            })

    boundary_heading = "#### Governance decisions, controls and evidence boundaries"
    if boundary_heading in text:
        boundary_text = text.split(boundary_heading, 1)[1].lower()
        boundary_terms = ["mandate", "trade-off", "tradeoff", "skills", "escalation", "assurance"]
        if not any(term in boundary_text for term in boundary_terms):
            warnings.append({
                "check": "governance_style_boundary_section_may_be_weak",
                "details": "Final Governance decisions/controls/boundaries subsection may not clearly include key evidence boundaries."
            })

    if re.search(r"IFRS\s*S?[12]?\s*§|§\s*\d|\[IFRS", text):
        failures.append({"check": "visible_ifrs_references", "details": "Visible IFRS paragraph references/tags found."})

    if "[REF:" in text or re.search(r"MTG-BANK\d+", text):
        failures.append({"check": "internal_refs_visible", "details": "Internal meeting references should not appear in final text."})

    required_numbers = {
        "missing_climate_agenda_pct": ("72.6", "72.6% climate agenda frequency is available and must be used."),
        "missing_board_climate_expertise_pct": ("37.5", "37.5% board climate expertise is available and must be used."),
        "missing_ceo_esg_remuneration_pct": ("9.3", "9.3% CEO ESG-linked remuneration is available and must be used."),
        "missing_all_exec_remuneration_pct": ("15.1", "15.1% all-executive climate-linked remuneration is available and must be used."),
    }
    for check_name, (num, detail) in required_numbers.items():
        if num not in text:
            failures.append({"check": check_name, "details": detail})

    required_trend_patterns = {
        "esg_committee_trend": ["5", "6", "7"],
        "board_expertise_trend": ["33.5", "35.5", "37.5"],
        "ceo_remuneration_trend": ["6.3", "7.8", "9.3"],
        "all_exec_remuneration_trend": ["8.7", "15.3", "15.1"],
        "climate_agenda_trend": ["69.8", "73.1", "72.6"],
        "board_meeting_trend": ["4", "7", "8"],
    }
    for check_name, values in required_trend_patterns.items():
        if not all(v in text for v in values):
            failures.append({"check": f"missing_{check_name}", "details": f"Expected trend values missing: {values}"})

    if not profile.get("formal_governance_mandate_available"):
        if _contains_any(text, ["formal mandate", "committee charter", "terms of reference"]) and not _contains_any(text, ["does not include", "not documented", "not available", "available documentation does not"]):
            failures.append({"check": "formal_mandate_overclaimed", "details": "Formal mandate/charter language used without a limitation."})

    if not profile.get("board_tradeoff_evidence_available"):
        if _contains_any(text, ["trade-off", "tradeoff"]) and not _contains_any(text, ["not describe", "not documented", "not available", "does not provide", "does not evidence", "documentation reviewed does not"]):
            failures.append({"check": "board_tradeoffs_overclaimed", "details": "Trade-off language used without direct evidence or limitation."})

    management_heading = "#### Management responsibility for climate-related risks and opportunities"
    next_heading = "#### Skills, competencies and remuneration"
    management_section = ""
    if management_heading in text:
        management_section = text.split(management_heading, 1)[1]
        if next_heading in management_section:
            management_section = management_section.split(next_heading, 1)[0]
    management_section_l = management_section.lower()

    if profile.get("management_risk_register_available"):
        required_groups = {
            "risk_register": ["risk register"],
            "risk_count": ["8 climate-related risks", "8 climate risks", "8 risks", "eight climate-related risks"],
            "categories": ["risk categor", "classified into categories", "physical", "transition"],
            "monitoring": ["monitoring", "quarterly", "semi-annual", "semi annual"],
            "scenario_links": ["scenario"],
            "mitigation": ["mitigation", "mitigat"],
            "erm": ["erm", "enterprise risk management"],
        }
        missing_groups = [
            group for group, terms in required_groups.items()
            if not any(term in management_section_l for term in terms)
        ]
        if missing_groups:
            failures.append({
                "check": "risk_register_process_omitted",
                "details": f"Risk-register evidence is available but the Management responsibility subsection is missing: {missing_groups}"
            })
    else:
        if _contains_any(text, ["risk register", "risk categories", "scenario links", "mitigation actions", "monitoring frequencies"]) and not _contains_any(text, ["does not contain", "not available", "not documented", "available evidence does not"]):
            failures.append({"check": "risk_register_process_invented", "details": "Risk-register process language appears although risk register is absent from Governance evidence."})

    if not profile.get("formal_escalation_thresholds_available"):
        escalation_terms = [
            "escalation threshold", "formal escalation", "escalation trigger",
            "trigger-based escalation", "defined escalation pathway", "escalation route"
        ]
        allowed_boundary = _contains_any(text, [
            "not documented", "not available", "does not specify", "does not evidence",
            "not evidenced", "no formal escalation",
            "available documentation does not evidence",
            "no such structures are therefore described"
        ])
        if any(term in text_l for term in escalation_terms) and not allowed_boundary:
            failures.append({"check": "escalation_threshold_overclaimed", "details": "Formal escalation language appears without evidence."})

    if not profile.get("skills_adequacy_process_available"):
        if _contains_any(text, ["skills adequacy assessment", "skills matrix", "skills gap analysis", "formal skills assessment"]) and not _contains_any(text, ["not documented", "not available", "does not describe"]):
            failures.append({"check": "skills_process_overclaimed", "details": "Formal skills adequacy process appears without evidence."})

    if profile.get("assurance_scope_is_scope1_scope2_only"):
        if "assurance" in text_l and "financed emissions" not in text_l:
            warnings.append({"check": "financed_emissions_scope_boundary_missing", "details": "Assurance section may not clearly state financed emissions are outside assurance scope."})
        if _contains_any(text, ["financed emissions are assured", "scope 3 emissions are assured", "assurance over financed emissions"]):
            failures.append({"check": "assurance_scope_overclaimed", "details": "Draft implies financed emissions / Scope 3 are assured."})

    forbidden_strong_phrases = [
        "governance oversight supports the bank's tcfd",
        "governance oversight supports the bank’s tcfd",
        "governance oversight ensures",
        "ensures alignment",
        "supports alignment",
        "fully aligned",
    ]
    found_strong = [p for p in forbidden_strong_phrases if p in text_l]
    if found_strong:
        failures.append({"check": "unsupported_alignment_or_strong_claim", "details": found_strong})

    return {
        "failures": failures,
        "warnings": warnings,
        "failure_count": len(failures),
        "warning_count": len(warnings),
    }


def writer_node(state: GovernanceState) -> GovernanceState:
    is_revision = state["revision_count"] > 0
    feedback = None

    if is_revision:
        judge = state.get("judge_result", {})
        issues = judge.get("required_fixes", [])
        checklist = judge.get("checklist", {})
        false_items = [k for k, v in checklist.items() if v is False and k != "false_count"]
        feedback = (
            "REQUIRED FIXES:\n" +
            "\n".join(f"- {fix}" for fix in issues) +
            "\n\nFAILED CHECKLIST ITEMS:\n" +
            "\n".join(f"- {item}" for item in false_items)
        )

    prompt = build_writer_prompt(state["evidence"], judge_feedback=feedback)

    draft = call_writer_llm(
        system_prompt=WRITER_SYSTEM,
        user_prompt=prompt,
    )

    print(f"\n{'='*50}")
    print(f"WRITER {'(revision ' + str(state['revision_count']) + ')' if is_revision else '(initial draft)'}")
    print(f"Draft length: {len(draft.split())} words")
    print(f"{'='*50}")

    return {
        **state,
        "draft": draft.strip(),
        "status": "judging"
    }


def judge_node(state: GovernanceState) -> GovernanceState:
    draft = state["draft"]
    deterministic_checks = run_governance_deterministic_checks(draft, state["evidence"])

    judge_prompt = build_judge_prompt(
        draft,
        state["evidence"],
        deterministic_checks=deterministic_checks,
    )
    judge_result = call_judge_llm_json(
        system_prompt=JUDGE_SYSTEM,
        user_prompt=judge_prompt,
    )
    judge_result = normalize_json_booleans(judge_result)

    judge_result.setdefault("approved", False)
    judge_result.setdefault(
        "approval_status",
        "approved" if judge_result.get("approved") else "revision_required",
    )
    judge_result.setdefault("main_issues", [])
    judge_result.setdefault("required_fixes", [])
    judge_result.setdefault("checklist", {})
    judge_result["deterministic_prechecks"] = deterministic_checks

    print("\nGOVERNANCE JUDGE RESULT — GPT-5.2 + DATA-AWARE PRECHECKS")
    print(json.dumps(judge_result, indent=2, ensure_ascii=False))

    return {
        **state,
        "judge_result": judge_result,
        "status": "judging",
    }


GOVERNANCE_REVISER_SYSTEM = """
You are a precise sustainability disclosure reviser.

Revise the existing Governance section using only:
- the supplied compact Governance evidence;
- the availability profile;
- the data-aware deterministic pre-checks; and
- the judge's required fixes.

Rules:
- Fix every judge issue and failed checklist item.
- Preserve correct content that was not criticised.
- Never invent evidence.
- If risk-register evidence is unavailable, do not describe a risk-register workflow.
- Keep the exact six-subsection Emirates-style Governance structure.
- Do not add visible IFRS paragraph references, meeting IDs or internal refs.
- Return only the complete revised Governance section using the required six headings.
""".strip()


def reviser_node(state: GovernanceState) -> GovernanceState:
    if state["revision_count"] >= state["max_revisions"]:
        return {**state, "status": "failed", "final_section": state["draft"]}

    judge = state.get("judge_result", {})
    issues = judge.get("required_fixes", [])
    checklist = judge.get("checklist", {})
    deterministic = judge.get("deterministic_prechecks", {})
    false_items = [k for k, v in checklist.items() if v is False and k != "false_count"]

    revision_prompt = f"""
STRICT GOVERNANCE REQUIREMENTS:
{IFRS_GOVERNANCE_REQUIREMENTS}

AVAILABLE-EVIDENCE PROFILE:
{json.dumps(state["evidence"].get("availability_profile", {}), indent=2, ensure_ascii=False)}

DETERMINISTIC PRE-CHECKS:
{json.dumps(deterministic, indent=2, ensure_ascii=False)}

COMPACT GOVERNANCE EVIDENCE:
{json.dumps(state["evidence"], indent=2, ensure_ascii=False)}

REVISION BOUNDARY:
- Use available evidence when the judge identifies an omission.
- When evidence is unavailable, preserve or improve the accurate boundary statement.
- Never invent a missing process, policy, threshold, trade-off, risk-register workflow, or assurance scope.

CURRENT DRAFT:
{state["draft"]}

JUDGE REQUIRED FIXES:
{json.dumps(issues, indent=2, ensure_ascii=False)}

FAILED CHECKLIST ITEMS:
{json.dumps(false_items, indent=2, ensure_ascii=False)}

Revise the current draft and return only the complete revised Governance section.
""".strip()

    revised_draft = call_reviser_llm(
        system_prompt=GOVERNANCE_REVISER_SYSTEM,
        user_prompt=revision_prompt,
    )

    new_revision_count = state["revision_count"] + 1
    print(f"\nGovernance revised with GPT-4.1 | revision {new_revision_count}")

    return {
        **state,
        "draft": revised_draft.strip(),
        "revision_count": new_revision_count,
        "status": "judging",
    }


def finalize_node(state: GovernanceState) -> GovernanceState:
    judge = state.get("judge_result", {})
    approved = judge.get("approved", False)

    print(f"\n{'='*50}")
    print(f"FINALIZED")
    print(f"  Status: {'APPROVED' if approved else 'MAX REVISIONS REACHED'}")
    print(f"  Final score: {judge.get('overall_score')}/10")
    print(f"  Revisions: {state['revision_count']}")
    print(f"{'='*50}")

    return {
        **state,
        "final_section": state["draft"],
        "status": "approved" if approved else "failed"
    }


def route_after_judge(state: GovernanceState) -> str:
    judge = state.get("judge_result", {})
    approved = judge.get("approved", False)
    revision_count = state.get("revision_count", 0)
    max_revisions = state.get("max_revisions", 2)

    if approved:
        return "finalize"
    if revision_count >= max_revisions:
        return "finalize"
    return "revise"

## 11. Governance LangGraph nodes and deterministic checks


In [ ]:
# ── BUILD AND COMPILE GRAPH ───────────────────────────────────
builder = StateGraph(GovernanceState)

builder.add_node("writer",   writer_node)
builder.add_node("judge",    judge_node)
builder.add_node("reviser",  reviser_node)
builder.add_node("finalize", finalize_node)

builder.add_edge(START,      "writer")
builder.add_edge("writer",   "judge")
builder.add_edge("reviser",  "judge")
builder.add_edge("finalize", END)

builder.add_conditional_edges(
    "judge",
    route_after_judge,
    {
        "revise":   "reviser",
        "finalize": "finalize",
    }
)

graph = builder.compile()
print("Graph compiled")

## 12. Build and compile Governance graph


In [ ]:
# ── RUN ──────────────────────────────────────────────────────
initial_state: GovernanceState = {
    "bank_name":      bank_name,
    "evidence":       evidence,
    "draft":          "",
    "judge_result":   {},
    "revision_count": 0,
    "max_revisions":  2,
    "status":         "drafting",
    "final_section":  "",
    "token_usage":    {}
}

print(f"Starting governance generation for: {bank_name}\n")
result = graph.invoke(initial_state)

## 13. Run Governance generation


In [ ]:
# ── OUTPUT ───────────────────────────────────────────────────
print("\n" + "="*60)
print("FINAL JUDGE RESULT")
print("="*60)
print(json.dumps(result["judge_result"], indent=2, ensure_ascii=False))

print("\n" + "="*60)
print("GOVERNANCE SECTION")
print("="*60)
print(result["final_section"])

# Save outputs
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

with open(output_dir / "governance_BANK01.md", "w", encoding="utf-8") as f:
    f.write(result["final_section"])

with open(output_dir / "governance_BANK01_meta.json", "w", encoding="utf-8") as f:
    json.dump({
        "bank_id":        "BANK01",
        "bank_name":      bank_name,
        "section":        "governance",
        "status":          result["status"],
        "approval_status": result["judge_result"].get("approval_status"),
        "raw_score":       result["judge_result"].get("raw_score_before_caps"),
        "final_score":     result["judge_result"].get("overall_score"),
        "score_cap_reason": result["judge_result"].get("score_cap_reason"),
        "revisions":       result["revision_count"],
        "approved":        result["judge_result"].get("approved"),
        "checklist":       result["judge_result"].get("checklist"),
        "issues":         result["judge_result"].get("main_issues"),
        "available_evidence_omitted": result["judge_result"].get("available_evidence_omitted"),
        "unsupported_claims": result["judge_result"].get("unsupported_claims"),
        "correctly_disclosed_evidence_boundaries": result["judge_result"].get("correctly_disclosed_evidence_boundaries"),
        "raw_governance_payload_path": str(RAW_GOVERNANCE_PATH),
        "compact_governance_evidence_path": str(COMPACT_GOVERNANCE_PATH),
    }, f, indent=2, ensure_ascii=False)

print(f"\nSaved to outputs/governance_BANK01.md")
print(f"Raw Governance payload saved to: {RAW_GOVERNANCE_PATH}")
print(f"Compact Governance evidence saved to: {COMPACT_GOVERNANCE_PATH}")
